In [1]:
# 分析対象銘柄の証券コードをセット
from datetime import date
code = 1301
get_latest_forcast = True
valuation_date = date.today()
# valuation_date = date(2022, 11, 18)

In [2]:
# 読み込みファイルパスの設定とimportしたいmoduleパス(pythonパス)の設定
from pathlib import Path
import os

CURRENT_DIR = Path(os.getcwd())
PJ_DIR = CURRENT_DIR.parent.parent
DATA_DIR = PJ_DIR / "data" 


# notebook内で利用するmoduleのimport
from wequant.data_processing import KessanPl, read_data, MeigaralistPl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pandas as pd
import polars as pl

In [3]:
fp1 = DATA_DIR / "kessan.parquet"
df1 = read_data(fp1)
KPL = KessanPl(df1)

In [4]:
df = KPL.get_target_stock_yearly_settlements(1301)

In [5]:
df

code,settlement_date,settlement_type,announcement_date,sales,operating_income,ordinary_profit,final_profit,reviced_eps,dividend,quater
i64,date,str,date,i64,i64,i64,i64,f64,f64,i64
1301,2017-03-31,"""本""",2017-05-11,236561,3723,3709,2422,230.7,60.0,-2
1301,2018-03-31,"""本""",2018-05-10,254783,4066,4437,3211,304.3,60.0,4
1301,2019-03-31,"""本""",2019-05-13,256151,3831,4434,2914,269.6,70.0,4
1301,2020-03-31,"""本""",2020-05-12,262519,2918,3608,2037,188.5,70.0,4
1301,2021-03-31,"""本""",2021-05-14,249197,4657,4879,3838,356.9,80.0,4
1301,2022-03-31,"""本""",2022-05-13,253575,6392,6904,4634,430.8,90.0,4
1301,2023-03-31,"""本""",2023-05-12,272167,8105,8182,5782,539.1,100.0,4
1301,2024-03-31,"""本""",2024-05-10,261604,8806,8856,5936,548.6,100.0,4
1301,2025-03-31,"""予""",2024-05-10,300000,10000,10000,7000,589.4,110.0,4


In [6]:

# def get_fig_yearly_settlement_trend_barchart(code: int, valuation_date: date=date.today(), get_latest_forcast=True) -> Figure:
fp1 = DATA_DIR / "kessan.parquet"
df1 = read_data(fp1)

fp2 = DATA_DIR / "meigaralist.parquet"
df2 = read_data(fp2)

# valuation_dateで絞り込み
df1 = df1.filter(pl.col("announcement_date")<=valuation_date)

KPL = KessanPl(df1)
self = KPL

# xを決算期の表記に変更
KPL.with_columns_financtial_period()
df = KPL.df
df = df.filter(pl.col("code")==code)\
    .filter(pl.col("settlement_type")=="本")

pandas_df = df.to_pandas()
sales_df = pandas_df[["決算期", "sales"]]

# 棒グラフを作成
# グラフ出力オプション
pio.renderers.default = 'iframe'

# 棒グラフのセット
graph_data = [
    go.Bar(
        x = sales_df["決算期"],
        y = sales_df["sales"],
        marker = dict(color="skyblue"),
        name = "売上高"
    )
]
fig = go.Figure(graph_data)

# 決算予想は、色を変える
# valuation_dateにおける最新forcastを追加
if get_latest_forcast:
    fdf = self.get_latest_yearly_settlements(
            reference_date=valuation_date,
            settlement_type="予"
    )
    fdf = fdf.filter(pl.col("code")==code)
    fdf = fdf.with_columns([
        (pl.col("決算期")+pl.lit("(予)")).alias("決算期")
    ])
    pandas_fdf = fdf.to_pandas()
    fig.add_trace(go.Bar(
        x = pandas_fdf["決算期"].iloc[-1:],
        y = pandas_fdf["sales"].iloc[-1:],
        name = "売上高(予)",
        marker = dict(color="lightpink")
        
    ))

# 利益率を右軸をy軸として、折れ線グラフでトレース
# 営業利益率をトレース
all_df = pl.concat([df, fdf])
column_idx = 0
label_idx = 1
color_idx = 2
line_trace_cols_attrs = [
    ['operating_income', '営業利益', 'orange'],
    ['ordinary_profit', '経常利益', 'lightgreen'],
    ['final_profit', '純利益', 'purple']
]

for a in line_trace_cols_attrs:
    fig.add_trace(go.Scatter(
        x=all_df['決算期'],
        y=all_df[a[column_idx]],
        mode='lines',
        name=a[label_idx],
        yaxis = 'y2',
        line=dict(color=a[color_idx], width=2)
    ))

# レイアウトの設定
MPL = MeigaralistPl(df2)
company_name = MPL.get_name(code)
fig.update_layout(
    title=f'{company_name}({code})業績推移',
    xaxis=dict(title='年度'),
    yaxis=dict(title='売上高 (百万円)'),
    yaxis2=dict(
        title="利益(百万円)",
        overlaying="y", # 左のY軸に重ねる
        side="right"
    ),
    bargap=0.2  # 棒の間隔
)

In [7]:
fdf

code,settlement_date,settlement_type,announcement_date,sales,operating_income,ordinary_profit,final_profit,reviced_eps,dividend,quater,yearly_settlement_date,fy,fm,決算期
i64,date,str,date,i64,i64,i64,i64,f64,f64,i64,date,str,str,str
1301,2025-03-31,"""予""",2024-05-10,300000,10000,10000,7000,589.4,110.0,4,2025-03-31,"""2025""","""3""","""2025年3月期(予)"""
